In [ ]:
import io
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import FileUpload
from IPython.display import clear_output, display

# ------------------------------
# Helper functions
# ------------------------------

def get_uploaded_file_content(upload_widget):
    """
    Safely extract uploaded image bytes from FileUpload,
    supporting both dict and tuple formats used by Voila.
    """
    if not upload_widget.value:
        return None
    
    val = upload_widget.value
    
    # Voila sometimes returns a dict
    if isinstance(val, dict):
        return list(val.values())[0]["content"]
    
    # Other times it returns a tuple
    if isinstance(val, tuple):
        return val[0]["content"]
    
    return None


MAX_SIZE = (512, 512)  # max width, height for resizing

def load_and_resize_image(content_bytes):
    """
    Load image from uploaded bytes and resize only if larger than MAX_SIZE,
    preserving aspect ratio.
    """
    img = Image.open(io.BytesIO(content_bytes))

    if img.width > MAX_SIZE[0] or img.height > MAX_SIZE[1]:
        img.thumbnail(MAX_SIZE)  # preserves aspect ratio
        print(f"Image resized to: {img.size}")
    else:
        print(f"Image kept at original size: {img.size}")

    return img, np.array(img)


# ------------------------------
# Widgets
# ------------------------------

upload = FileUpload(
    accept='image/*',
    multiple=False,
    description="Upload Image"
)

cutoff_slider = widgets.IntSlider(
    value=20, min=1, max=100, step=1,
    description="Filter strength"
)

out_original = widgets.Output()
out_fft = widgets.Output()
out_filtered = widgets.Output()


# ------------------------------
# Update function
# ------------------------------

def update_plot(change=None):
    # Clear existing outputs
    out_original.clear_output(wait=True)
    out_fft.clear_output(wait=True)
    out_filtered.clear_output(wait=True)

    # Get uploaded file
    content = get_uploaded_file_content(upload)
    if content is None:
        with out_original:
            print("Please upload an image.")
        return

    try:
        # Load and resize if needed
        pil_img, img_array = load_and_resize_image(content)

        # Show original
        with out_original:
            plt.figure(figsize=(4,4))
            plt.imshow(pil_img, cmap='gray')
            plt.title("Uploaded Image")
            plt.axis('off')
            plt.show()

        # Compute FFT
        f = np.fft.fft2(img_array)
        fshift = np.fft.fftshift(f)
        magnitude_spectrum = np.log(1 + np.abs(fshift))

        with out_fft:
            plt.figure(figsize=(4,4))
            plt.imshow(magnitude_spectrum, cmap='gray')
            plt.title("FFT Magnitude Spectrum")
            plt.axis('off')
            plt.show()

        # Low-pass filter
        rows, cols = img_array.shape[:2] if img_array.ndim == 2 else img_array.shape[:2]
        crow, ccol = rows // 2, cols // 2
        cutoff = cutoff_slider.value

        mask = np.zeros((rows
